# Reproduce the Gap — decodability ≠ steerability, in 5 minutes (CPU-only)

**The Decodability Gap:** a frozen language model can *decode* the right answer far above chance, yet no static steering intervention *changes* its decisions. Decoding is not steering. This notebook reproduces that dissociation live on a toy split of the SCBI cloze bench.

**What you'll see**
1. **DECODE** — Pythia-160m answers a 12-item cloze bench at well above the 12.5% chance rate.
2. **STEER** — the pre-registered intervention battery (mask the top-1 / foil / random candidate, re-argmax) barely moves decisions: ΔM ≈ 0.
3. **The dissociation**, printed plainly and plotted: *"Decodes at X× chance. Steers zero decisions."*

**The full falsification kit** (60-item bench, Tier-0 runner, full battery with the binding decision table, one-command reproduction of the published EXP105 numbers) lives at `falsification_kit/` in the SCBI repo — this notebook is the 5-minute demo, not the rig.

⏱ Expected runtime on a free Colab CPU: **under 5 minutes** (pip install + 160m download dominate).

In [ ]:
# ── Cell 1: environment ─────────────────────────────────────────────
# Installs deps if missing (works in Colab AND as a plain script).
import importlib.util, subprocess, sys, os

def _need(pkg):
    return importlib.util.find_spec(pkg) is None

_missing = [p for p in ("torch", "transformers", "numpy", "matplotlib") if _need(p)]
if _missing:
    print("installing:", _missing, flush=True)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "torch", "--index-url", "https://download.pytorch.org/whl/cpu",
        "transformers", "numpy", "matplotlib",
    ])

import time, math, random, hashlib
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Demo knobs (override via env for headless validation):
MODEL_ID = os.environ.get("GAP_MODEL_ID", "EleutherAI/pythia-160m")
TOY_N    = int(os.environ.get("GAP_TOY_N", "12"))
SEED     = 20260928
torch.manual_seed(SEED); random.seed(SEED)

print("torch", torch.__version__, "| model:", MODEL_ID, "| toy items:", TOY_N, flush=True)

## Step 0 — the toy bench

The full kit bench has 60 cloze items (8 candidates each, chance = 1/8 = 12.5%). For this demo we embed a **deterministic stride-5 sample** — items 0, 5, 10, …, 55 — copied verbatim from `falsification_kit/bench/cloze_bench.json`. Stride sampling covers the bench's topic spread without cherry-picking. The toy split is a *demo*, not a verdict: binding numbers come from the full bench (see the scope note at the end).

In [ ]:
# ── Cell 2: toy bench (verbatim from falsification_kit/bench/cloze_bench.json) ──
# Slimmed copies of falsification_kit/core/scoring.py — same algorithms,
# trimmed for the demo (the full rig also has Tango CIs, the tie-guard,
# and the binding Tier-0/battery bar orders).

TOY_ITEMS = [
  {
    "item_id": "exp104_t0_00",
    "question": "The Eiffel Tower stands in the capital of which country?",
    "prompt": "The Eiffel Tower stands in the capital of which country? The answer is",
    "candidates": [
      "Germany",
      "Italy",
      "Austria",
      "Greece",
      "Spain",
      "France",
      "Portugal",
      "Belgium"
    ],
    "correct_idx": 5,
    "foil_idx": 1
  },
  {
    "item_id": "exp104_t0_05",
    "question": "The Little Mermaid statue stands in the capital of which country?",
    "prompt": "The Little Mermaid statue stands in the capital of which country? The answer is",
    "candidates": [
      "Iceland",
      "Denmark",
      "Germany",
      "Netherlands",
      "Norway",
      "Belgium",
      "Sweden",
      "Finland"
    ],
    "correct_idx": 1,
    "foil_idx": 6
  },
  {
    "item_id": "exp104_t0_10",
    "question": "The company that makes the Beetle car is headquartered in which country?",
    "prompt": "The company that makes the Beetle car is headquartered in which country? The answer is",
    "candidates": [
      "Japan",
      "Spain",
      "Sweden",
      "France",
      "Poland",
      "Austria",
      "Italy",
      "Germany"
    ],
    "correct_idx": 7,
    "foil_idx": 5
  },
  {
    "item_id": "exp104_t0_15",
    "question": "The film Pan's Labyrinth was directed by a person born in which country?",
    "prompt": "The film Pan's Labyrinth was directed by a person born in which country? The answer is",
    "candidates": [
      "Brazil",
      "Colombia",
      "Italy",
      "Mexico",
      "Argentina",
      "Portugal",
      "Chile",
      "Spain"
    ],
    "correct_idx": 3,
    "foil_idx": 7
  },
  {
    "item_id": "exp104_t0_20",
    "question": "The Spree flows through the capital of which country?",
    "prompt": "The Spree flows through the capital of which country? The answer is",
    "candidates": [
      "Germany",
      "Croatia",
      "Serbia",
      "Poland",
      "Romania",
      "Hungary",
      "Austria",
      "Belgium"
    ],
    "correct_idx": 0,
    "foil_idx": 6
  },
  {
    "item_id": "exp104_t0_25",
    "question": "The museum that houses Guernica is in which city?",
    "prompt": "The museum that houses Guernica is in which city? The answer is",
    "candidates": [
      "Berlin",
      "Lisbon",
      "Vienna",
      "Rome",
      "Amsterdam",
      "Madrid",
      "Barcelona",
      "Paris"
    ],
    "correct_idx": 5,
    "foil_idx": 6
  },
  {
    "item_id": "exp104_t0_30",
    "question": "The novel One Hundred Years of Solitude was written by an author born in which country?",
    "prompt": "The novel One Hundred Years of Solitude was written by an author born in which country? The answer is",
    "candidates": [
      "Venezuela",
      "Peru",
      "Chile",
      "Colombia",
      "Argentina",
      "Mexico",
      "Ecuador",
      "Brazil"
    ],
    "correct_idx": 3,
    "foil_idx": 5
  },
  {
    "item_id": "exp104_t0_35",
    "question": "The novel The Shadow of the Wind was written by an author born in which country?",
    "prompt": "The novel The Shadow of the Wind was written by an author born in which country? The answer is",
    "candidates": [
      "Italy",
      "Argentina",
      "France",
      "Spain",
      "Greece",
      "Mexico",
      "Chile",
      "Portugal"
    ],
    "correct_idx": 3,
    "foil_idx": 5
  },
  {
    "item_id": "exp104_t0_40",
    "question": "The founder of Disney was born in which US state?",
    "prompt": "The founder of Disney was born in which US state? The answer is",
    "candidates": [
      "Texas",
      "Illinois",
      "California",
      "Indiana",
      "Florida",
      "Ohio",
      "Wisconsin",
      "Missouri"
    ],
    "correct_idx": 1,
    "foil_idx": 2
  },
  {
    "item_id": "exp104_t0_45",
    "question": "The 2008 Summer Olympics were held in which country?",
    "prompt": "The 2008 Summer Olympics were held in which country? The answer is",
    "candidates": [
      "Philippines",
      "Japan",
      "Thailand",
      "Korea",
      "Taiwan",
      "China",
      "Vietnam",
      "Indonesia"
    ],
    "correct_idx": 5,
    "foil_idx": 1
  },
  {
    "item_id": "exp104_t0_50",
    "question": "The Treaty of Utrecht was signed in which country?",
    "prompt": "The Treaty of Utrecht was signed in which country? The answer is",
    "candidates": [
      "Belgium",
      "Austria",
      "France",
      "Sweden",
      "Germany",
      "Netherlands",
      "Poland",
      "Denmark"
    ],
    "correct_idx": 5,
    "foil_idx": 0
  },
  {
    "item_id": "exp104_t0_55",
    "question": "The painting Guernica hangs in a museum in which country?",
    "prompt": "The painting Guernica hangs in a museum in which country? The answer is",
    "candidates": [
      "Belgium",
      "France",
      "Greece",
      "Italy",
      "Austria",
      "Spain",
      "Germany",
      "Portugal"
    ],
    "correct_idx": 5,
    "foil_idx": 1
  }
]

TOY_ITEMS = TOY_ITEMS[:TOY_N]
K = 8  # candidates per item

def correct_rank(logits8, correct_idx, token_ids8):
    order = sorted(range(K), key=lambda j: (-logits8[j], token_ids8[j]))
    return order.index(correct_idx) + 1

def top1_decision(logits8, token_ids8):
    return max(range(K), key=lambda j: (logits8[j], -token_ids8[j]))

def masked_argmax(logits8, mask_idx, token_ids8):
    m = list(logits8); m[mask_idx] = -1e9
    return top1_decision(m, token_ids8)

def mcnemar_exact(b, c):
    n = b + c
    if n == 0: return 1.0
    k = min(b, c)
    tail = sum(math.comb(n, i) for i in range(k + 1)) / (2 ** n)
    return min(1.0, 2.0 * tail)

def battery_arms(logits8, tids, top1, correct, foil, rng):
    arms = {"B": top1_decision(logits8, tids)}
    arms["T"] = masked_argmax(logits8, top1, tids)     # mask baseline top-1
    arms["F"] = masked_argmax(logits8, foil, tids)     # mask structural foil
    elig = [i for i in range(K) if i != top1 and i != correct]
    arms["R"] = masked_argmax(logits8, rng.choice(elig), tids)  # random mask
    return arms

print(f"toy bench loaded: {len(TOY_ITEMS)} items × {K} candidates")

## Step 1 — DECODE: does the model know the answer?

We load Pythia-160m (the smallest Pythia — fastest honest demo; the published EXP105 numbers used the 410m sibling with the same tokenizer) and score each item: the model's top-1 candidate vs the correct one. First the **G1 gate**: every candidate string must encode to exactly one token, otherwise the item is excluded — a readout benchmark is only licensed on items that pass.

In [ ]:
# ── Cell 3: load model + G1 gate + score ─────────────────────────────
t_all = time.time()
tok = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
model.eval()
print(f"model loaded in {time.time()-t_all:.0f}s | params: {sum(p.numel() for p in model.parameters())/1e6:.0f}M", flush=True)

# Δθ=0 guard: hash weights before scoring (frozen backbone — inference only)
def _whash(m):
    h = hashlib.sha256()
    with torch.no_grad():
        for p in m.parameters():
            h.update(p.detach().to(torch.float32).cpu().numpy().tobytes())
    return h.hexdigest()
hash_before = _whash(model)

records, g1_fail = [], []
for it in TOY_ITEMS:
    tids, ok = [], True
    for cand in it["candidates"]:
        ids = tok.encode(" " + cand, add_special_tokens=False)
        if len(ids) != 1: ok = False; break
        tids.append(ids[0])
    if not ok or len(set(tids)) != K:
        g1_fail.append(it["item_id"]); continue
    with torch.no_grad():
        logits = model(tok(it["prompt"], return_tensors="pt")["input_ids"]).logits[0, -1, :].float()
    cl = [float(logits[t]) for t in tids]
    top1 = top1_decision(cl, tids)
    rank = correct_rank(cl, it["correct_idx"], tids)
    records.append(dict(item_id=it["item_id"], candidates=it["candidates"],
                        correct_idx=it["correct_idx"], foil_idx=it["foil_idx"],
                        top1_idx=top1, correct_rank=rank,
                        candidate_logits=cl, candidate_token_ids=tids))

assert _whash(model) == hash_before, "WEIGHT DRIFT"
print(f"G1 gate: {len(records)}/{len(TOY_ITEMS)} toy items passed single-token attestation")
print("Δθ=0 guard: weight hash identical pre/post ✔")

n_correct = sum(r["top1_idx"] == r["correct_idx"] for r in records)
acc = n_correct / len(records)
print(f"\nDECODE accuracy: {n_correct}/{len(records)} = {acc:.3f}  vs  chance {1/K:.3f}  →  {acc/(1/K):.1f}× chance")
for r in records:
    mark = "✓" if r["top1_idx"] == r["correct_idx"] else "✗"
    print(f"  {mark} {r['item_id']}: top1={r['candidates'][r['top1_idx']]:<10} correct={r['candidates'][r['correct_idx']]:<10} rank={r['correct_rank']}")

## Step 2 — STEER: can we change its decisions?

Decoding works. Now the falsification question: does *intervening* move decisions? We run the kit's pre-registered battery arms — pure logit operators, zero extra model contact — on the toy population (failed items with correct-rank ≤ 5):

- **B** baseline (no intervention) · **T** mask the model's top-1 · **F** mask the structural foil · **R** mask a random candidate
- ΔM = fraction of population items rescued (flipped to correct) minus baseline
- The binding contrast is **T-vs-R** (truth-masking vs random-masking, McNemar-exact, α = 0.05)

In [ ]:
# ── Cell 4: intervention battery on the toy records ──────────────────
rng = random.Random(SEED)
pop = [r for r in records
       if r["top1_idx"] != r["correct_idx"] and r["correct_rank"] <= 5]

if not pop:
    print("toy population empty (model aced the toy split or nothing in rank≤5) — steering demo skipped")
    dM_T = dM_F = dM_R = 0.0; p_TR = 1.0; n_pop = 0
else:
    n_pop = len(pop)
    arm_ok = {a: 0 for a in "BTFR"}
    b = c = 0  # McNemar T-vs-R discordant pairs
    for r in pop:
        arms = battery_arms(r["candidate_logits"], r["candidate_token_ids"],
                            r["top1_idx"], r["correct_idx"], r["foil_idx"], rng)
        ok = {a: idx == r["correct_idx"] for a, idx in arms.items()}
        for a in "BTFR": arm_ok[a] += ok[a]
        if ok["T"] and not ok["R"]: b += 1
        if ok["R"] and not ok["T"]: c += 1
    base = arm_ok["B"]
    dM_T, dM_F, dM_R = ((arm_ok[a] - base) / n_pop for a in "TFR")
    p_TR = mcnemar_exact(b, c)
    print(f"toy population n_pop={n_pop} (failed, rank≤5)")
    print(f"rescues: B={arm_ok['B']} T={arm_ok['T']} F={arm_ok['F']} R={arm_ok['R']}")
    print(f"ΔM_T={dM_T:+.3f}  ΔM_F={dM_F:+.3f}  ΔM_R={dM_R:+.3f}")
    print(f"T-vs-R McNemar: b={b} c={c} p={p_TR:.4f}  (binding bar: p<0.05)")
    print("note: toy n_pop is far below the n_pop≥12 gate — this is DESCRIPTIVE, not a verdict")

## Step 3 — the dissociation, plotted

In [ ]:
# ── Cell 5: plot + the one-line summary ──────────────────────────────
import matplotlib
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].bar(["chance (1/8)", "Pythia-160m\ndecode"], [1/K, acc],
          color=["#bbbbbb", "#2a7fbf"])
ax[0].set_ylim(0, 1); ax[0].set_ylabel("accuracy")
ax[0].set_title("DECODE: the model knows")
for i, v in enumerate([1/K, acc]): ax[0].text(i, v + 0.02, f"{v:.3f}", ha="center")

ax[1].bar(["ΔM_T\n(mask top-1)", "ΔM_R\n(mask random)"], [dM_T, dM_R],
          color=["#c65b3f", "#bbbbbb"])
ax[1].axhline(0, color="black", linewidth=0.8)
ax[1].set_ylabel("ΔM (rescue rate vs baseline)")
ax[1].set_title("STEER: interventions move ~nothing")

fig.suptitle("The Decodability Gap — reproduced live", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("/tmp/gap_chart.png", dpi=110)
plt.show()

x_times = acc / (1/K)
steer_word = "zero decisions" if abs(dM_T) < 0.05 and abs(dM_R) < 0.05 else "almost nothing"
print(f"\n  Decodes at {x_times:.1f}× chance. Steers {steer_word}.")
print(f"  (decode acc={acc:.3f} vs chance={1/K:.3f}; ΔM_T={dM_T:+.3f}, ΔM_R={dM_R:+.3f}, T-vs-R p={p_TR:.4f})")
print(f"\nTotal wall time: {time.time()-t_all:.0f}s")

## Honest scope — what this does and doesn't show

**What you just saw.** On a 12-item toy split, a frozen Pythia-160m decoded answers far above chance while the pre-registered intervention battery barely moved decisions. That is the Decodability Gap in miniature.

**What this is NOT.**
- Not a verdict: the toy `n_pop` is far below the binding `n_pop ≥ 12` gate, so no KILL/CONTINUE/PIVOT claim is licensed from this notebook. The binding result is **EXP105 on the full 60-item bench** (Pythia-410m, LOG-4491): baseline **32/60 = 0.5333** vs 0.125 chance; battery `n_pop=21`, truth-masking ΔM_T=+0.2381 (5/21 rescues), random-masking ΔM_R=0.0 (0/21), primary T-vs-R McNemar **p=0.0625** — failing the pre-registered p<0.05 bar → **Refuted (KILL, step 9)**. One command reproduces it bit-for-bit: `bash falsification_kit/reproduce_exp105.sh`.
- Not a steering method: masking is a *falsification instrument*, not a capability. A null here doesn't prove steering is impossible — it proves *this* static intervention family failed *this* pre-registered test.
- Not general: one model family (Pythia), one bench, N=60. The gap is a boundary measurement, not a universal law.

**Go further:** the full kit (`falsification_kit/`) — `tier0.py` runs the 60-item bench on any HuggingFace causal LM, `battery.py` walks the binding decision table, `tests/` carry the seven degeneracy unit tests. If you run it on your own model, you become a replication.